# Cornell Chronicle: collect AI article links



In [ ]:
from pathlib import Path
from time import sleep
import random
from urllib.parse import urljoin, urlparse, urlunparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
BASE_URL = "https://news.cornell.edu"
CATEGORY_URL = f"{BASE_URL}/categories/artificial-intelligence"

DATA_DIR = Path("./Data")
OUTPUT_PATH = DATA_DIR / "Cornell Articles Links.csv"

# The category currently has far fewer pages, but this upper bound allows
# the notebook to continue working if Cornell adds more pages.
MAX_PAGES = 200

REQUEST_TIMEOUT = 30
DELAY_RANGE = (0.8, 1.5)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Create a persistent session with automatic retry handling.
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
def clean_text(value):
    """Collapse repeated whitespace and return a clean string."""
    if value is None:
        return ""
    return " ".join(str(value).split())


def normalize_story_url(href):
    """Convert a relative Cornell URL into one canonical absolute URL."""
    absolute = urljoin(BASE_URL, href)
    parsed = urlparse(absolute)

    # Cornell sometimes exposes equivalent /index.php/stories/... URLs.
    path = parsed.path.replace("/index.php/stories/", "/stories/")

    normalized = parsed._replace(
        scheme="https",
        netloc="news.cornell.edu",
        path=path.rstrip("/"),
        params="",
        query="",
        fragment="",
    )
    return urlunparse(normalized)


def extract_article_links(soup):
    """Extract title links from a Cornell category page.

    Cornell currently renders article titles as h2 links. Several fallback
    selectors are retained so minor Drupal theme changes do not immediately
    break the scraper.
    """
    selectors = [
        "main h2 a[href]",
        ".view-content h2 a[href]",
        "main h3 a[href]",
        ".view-content h3 a[href]",
    ]

    candidates = []
    for selector in selectors:
        candidates.extend(soup.select(selector))

    # Last-resort fallback: collect links whose URL contains /stories/.
    if not candidates:
        candidates = soup.select('main a[href*="/stories/"], a[href*="/stories/"]')

    records_by_url = {}

    for anchor in candidates:
        href = anchor.get("href", "").strip()
        if not href:
            continue

        url = normalize_story_url(href)
        parsed = urlparse(url)

        # Retain only Cornell Chronicle story pages.
        if parsed.netloc != "news.cornell.edu" or "/stories/" not in parsed.path:
            continue

        title = clean_text(anchor.get_text(" ", strip=True))
        if not title:
            image = anchor.find("img")
            title = clean_text(image.get("alt", "")) if image else ""

        # A story can appear through both an image link and a title link.
        # Prefer the non-empty title, then the longer title.
        old = records_by_url.get(url)
        if old is None or (title and len(title) > len(old["Article"])):
            records_by_url[url] = {"Article": title, "Link": url}

    return list(records_by_url.values())


def fetch_category_page(page_number):
    """Request one zero-indexed category page."""
    page_url = CATEGORY_URL if page_number == 0 else f"{CATEGORY_URL}?page={page_number}"

    response = session.get(page_url, timeout=REQUEST_TIMEOUT)

    if response.status_code == 404:
        return None, page_url, 404

    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser"), page_url, response.status_code

In [ ]:
# -----------------------------
# Crawl all category pages
# -----------------------------
all_records = []
seen_urls = set()

for page_number in tqdm(range(MAX_PAGES), desc="Category pages"):
    try:
        soup, page_url, status_code = fetch_category_page(page_number)
    except requests.RequestException as exc:
        print(f"Stopped at page {page_number}: {exc}")
        break

    if soup is None:
        print(f"Stopped: page {page_number} returned HTTP {status_code}.")
        break

    page_records = extract_article_links(soup)
    new_records = [row for row in page_records if row["Link"] not in seen_urls]

    # A page beyond the final page may be empty or repeat an earlier page.
    if not new_records:
        print(f"Stopped at page {page_number}: no new article links found.")
        break

    for row in new_records:
        row["Category Page"] = page_url
        row["Page Number"] = page_number
        all_records.append(row)
        seen_urls.add(row["Link"])

    # Incremental checkpoint: useful if the crawl is interrupted.
    checkpoint = pd.DataFrame(all_records)
    checkpoint.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print(
        f"Page {page_number}: {len(new_records)} new links; "
        f"{len(all_records)} unique links in total."
    )
    sleep(random.uniform(*DELAY_RANGE))

In [ ]:
article_links_df = (
    pd.DataFrame(all_records)
    .drop_duplicates(subset="Link")
    .reset_index(drop=True)
)

article_links_df.head()

In [ ]:
article_links_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(f"Saved {len(article_links_df)} unique article links to: {OUTPUT_PATH.resolve()}")